# MiniT2I general model, training on Colab

This notebook trains the general text to image model from scratch. It uses the CC3M photos (2.9 million captioned images), the Stable Diffusion VAE and a multilingual CLIP text encoder, both frozen. The 120 million parameter transformer is the part that learns.

How to use

1. Runtime, Change runtime type, T4 GPU.
2. Run all cells. The first one asks for permission to use your Google Drive, where the prepared data lives.
3. The second cell prepares the dataset in the background and the third one trains at the same time, using the shards that are already done.
4. When the session ends, open the notebook again and run everything. Training and preparation both resume where they stopped.

Every 30 minutes a grid of samples is saved to minit2i/amostras in your Drive.

With an HF_TOKEN secret (key icon on the left), checkpoints go to a private Hugging Face repository, the web model is updated after every save and the live dashboard receives real training samples every two minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
PASTA = '/content/drive/MyDrive/minit2i'
DADOS = f'{PASTA}/dados'
os.makedirs(DADOS, exist_ok=True)
if os.path.exists('/content/minit2i'):
    !git -C /content/minit2i pull -q
else:
    !git clone -q https://github.com/Samuelpedroso-dev/minit2i /content/minit2i
!pip install -q diffusers sentence-transformers onnx onnxruntime
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Hugging Face token loaded, checkpoints and the web model go there')
except Exception as e:
    print('no Hugging Face token, checkpoints stay in Google Drive', e.__class__.__name__)
print('ready,', len(os.listdir(DADOS)), 'shards already prepared')

In [ ]:
# dataset preparation in the background, 200 shards, about 1 million images
log = open(f'{PASTA}/preparo.log', 'a')
prep = subprocess.Popen(['python', 'preparar.py', '--saida', DADOS, '--inicio', '0', '--fim', '200'],
                        cwd='/content/minit2i/geral', stdout=log, stderr=subprocess.STDOUT)
print('preparation running, follow it in minit2i/preparo.log')

In [ ]:
# training, waits for the first 10 shards, saves every 30 minutes
!git -C /content/minit2i pull -q && cd /content/minit2i/geral && python treinar.py --dados {DADOS} --ckpt {PASTA}/geral.pt --minutos 600

In [ ]:
from IPython.display import Image, display
display(Image(f'{PASTA}/amostras/ultima.png', width=768))

In [ ]:
!tail -5 {PASTA}/preparo.log